# Cross-Regime Ablation Study: Leave-One-Out Validation

**Objective**: Systematically evaluate model architectures using **leave-one-regime-out cross-validation** to measure true generalization across market conditions.

**Why This Matters**: Training and testing within the same regime only proves the model can memorize patterns. To validate architectural choices, we must test whether models trained on diverse historical data can generalize to completely unseen market regimes.

---

## Methodology: Leave-One-Regime-Out Cross-Validation

For each of 8 market regimes (2018-2026):
1. **Train** on 7 regimes (all except the held-out)
2. **Test** on 1 held-out regime
3. Record performance metrics

This yields **8 independent test scores** per variant, enabling:
- Robust statistics (mean ± std)
- Regime-specific insights (which architectures fail on crashes vs low-vol)
- Quantified robustness (lower variance = more generalizable)

**Total Training Runs**: Multiple architecture variants × 8 folds each

---

## Ablation Studies

### Study 1: Component Ablation
**Question**: Does the ensemble (PatchTST + GNN) generalize better than individual components?

**Variants**:
- `patchtst`: Temporal-only baseline (learns time-series patterns)
- `gnn`: Cross-sectional-only baseline (learns option surface structure)
- `ensemble`: Combined model (both temporal and cross-sectional)

**Hypothesis**: Ensemble should have highest mean AND lowest variance (robustness)

### Study 2: Learnable Edge Weights
**Question**: Do learnable edges improve robustness across regimes?

**Variants**:
- `ensemble` (fixed edges based on moneyness/DTE distance)
- `ensemble --learnable-edges` (edges learned from data)

**Hypothesis**: Learnable edges adapt to regime-specific surface dynamics → lower variance

---

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import product

from src.models.repository import ResultsRepository

# Plotting config
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['font.size'] = 11

print("✓ Imports loaded")

## Regime Definitions

Define the 8 distinct market regimes spanning 2018-2026. Each represents a different macro environment.

In [ ]:
REGIMES = [
    {"id": 0, "label": "2018_H2", "start": "2018-07-01", "end": "2018-12-31",
     "description": "Vol spike (VIX >50), trade war"},
    {"id": 1, "label": "2019", "start": "2019-01-01", "end": "2019-12-31",
     "description": "Recovery rally, rate cuts"},
    {"id": 2, "label": "2020_H1", "start": "2020-01-01", "end": "2020-06-30",
     "description": "COVID crash + V-recovery"},
    {"id": 3, "label": "2021", "start": "2021-01-01", "end": "2021-12-31",
     "description": "Meme stocks, low vol grind"},
    {"id": 4, "label": "2022_H1", "start": "2022-01-01", "end": "2022-06-30",
     "description": "Bear market start, rate hikes"},
    {"id": 5, "label": "2023_H1", "start": "2023-01-01", "end": "2023-06-30",
     "description": "Banking crisis, recovery"},
    {"id": 6, "label": "2024_H2", "start": "2024-07-01", "end": "2024-12-31",
     "description": "Election, Aug vol spike (VIX >65)"},
    {"id": 7, "label": "2025", "start": "2025-01-01", "end": "2025-12-31",
     "description": "Recent period"},
]

print(f"Defined {len(REGIMES)} regimes spanning 2018-2026:\n")
for r in REGIMES:
    print(f"  [{r['id']}] {r['label']:12} {r['start']} to {r['end']} - {r['description']}")

## Training Commands

Before running analysis, execute leave-one-regime-out training for all variants.

### Helper: Build Training Date Ranges

For each held-out regime, construct training periods from the other 7 regimes.

In [ ]:
def get_training_periods(holdout_id):
    """Get train/val/test periods for leave-one-out fold.
    
    Returns the 4-boundary split args matching the train_model.py CLI:
      --train-start, --val-start, --test-start, --test-end
    
    Train runs from train_start to val_start; val from val_start to test_start;
    test from test_start to test_end.
    """
    test_regime = REGIMES[holdout_id]
    train_regimes = [r for r in REGIMES if r['id'] != holdout_id]
    
    # Train on all non-held-out regimes
    # For simplicity, use last regime in training set for validation
    val_regime = train_regimes[-1]
    pure_train_regimes = train_regimes[:-1]
    
    return {
        'train_regimes': pure_train_regimes,
        'val_regime': val_regime,
        'test_regime': test_regime,
        # 4-boundary split (matches train_model.py CLI)
        'train_start': pure_train_regimes[0]['start'],
        'val_start': val_regime['start'],
        'test_start': test_regime['start'],
        'test_end': test_regime['end'],
    }

# Example: holdout regime 2 (2020 COVID)
example = get_training_periods(2)
print("Example: Hold out 2020_H1 (COVID crash)\n")
print(f"--train-start {example['train_start']}")
print(f"--val-start   {example['val_start']}")
print(f"--test-start  {example['test_start']}")
print(f"--test-end    {example['test_end']}")
print(f"\nTrain regimes: {len(example['train_regimes'])}")
print(f"Val regime:    {example['val_regime']['label']}")
print(f"Test regime:   {example['test_regime']['label']}")

### Example Training Commands

The training script uses 4-boundary splits: `--train-start`, `--val-start`, `--test-start`, `--test-end`.
Train runs from train_start to val_start, val from val_start to test_start, test from test_start to test_end.

For **holdout regime 2** (2020_H1 COVID crash):

```bash
# Component ablation variants
python scripts/train_model.py \
    --train-start 2018-07-01 --val-start 2024-07-01 \
    --test-start 2020-01-01 --test-end 2020-06-30 \
    --ablation patchtst --epochs 50 --device cuda

python scripts/train_model.py \
    --train-start 2018-07-01 --val-start 2024-07-01 \
    --test-start 2020-01-01 --test-end 2020-06-30 \
    --ablation gnn --epochs 50 --device cuda

python scripts/train_model.py \
    --train-start 2018-07-01 --val-start 2024-07-01 \
    --test-start 2020-01-01 --test-end 2020-06-30 \
    --ablation ensemble --epochs 50 --device cuda

# Learnable edges variant
python scripts/train_model.py \
    --train-start 2018-07-01 --val-start 2024-07-01 \
    --test-start 2020-01-01 --test-end 2020-06-30 \
    --ablation ensemble --learnable-edges --epochs 50 --device cuda
```

**Note**: The 4-boundary split model assumes contiguous periods (train→val→test).
For leave-one-out where the held-out regime falls *within* the training window,
the training set will include the held-out period — a true leave-one-out implementation
would require non-contiguous split support (future enhancement).

**Repeat for all 8 holdout regimes** (0-7).

**Total**: 3 component variants × 8 folds + 1 learnable variant × 8 folds = **32 training runs**

**Estimated time**: ~2-3 hours/run × 32 = **64-96 hours** (2-4 days on single GPU)

---

## Load Results

In [ ]:
repo = ResultsRepository()
all_runs = repo.list()

if all_runs.empty:
    print("⚠️  No training runs found. Complete training first (see commands above).")
else:
    print(f"✓ Found {len(all_runs)} runs\n")
    
    # Parse test period from run name to identify held-out regime
    # Assumes naming convention includes test dates
    all_runs['test_start'] = pd.to_datetime(all_runs['test_start'])
    
    # Match test period to regime
    def match_regime(test_start):
        for r in REGIMES:
            if test_start >= pd.to_datetime(r['start']) and test_start <= pd.to_datetime(r['end']):
                return r['label']
        return 'Unknown'
    
    all_runs['holdout_regime'] = all_runs['test_start'].apply(match_regime)
    
    print("Summary by variant:")
    summary = all_runs.groupby('ablation').agg({
        'name': 'count',
        'best_metric': ['mean', 'std', 'min', 'max']
    })
    print(summary.round(4))
    
    print("\nExpected: 8 runs per variant (one per held-out regime)")
    for abl in ['patchtst', 'gnn', 'ensemble']:
        count = len(all_runs[all_runs['ablation'] == abl])
        status = "✓" if count == 8 else "⚠️"
        print(f"  {status} {abl:10} {count}/8 runs")

---

## Component Ablation Analysis

In [ ]:
component_runs = all_runs[all_runs['ablation'].isin(['patchtst', 'gnn', 'ensemble'])]

if component_runs.empty:
    print("⚠️  No component ablation runs found.")
else:
    print("=" * 70)
    print("COMPONENT ABLATION - CROSS-REGIME GENERALIZATION")
    print("=" * 70)
    
    stats = component_runs.groupby('ablation')['best_metric'].agg(['count', 'mean', 'std', 'min', 'max'])
    stats = stats.sort_values('mean', ascending=False)
    
    print("\nPerformance across 8 held-out regimes (Rank IC):\n")
    print(stats.round(4))
    
    print("\n" + "-" * 70)
    best_mean = stats['mean'].idxmax()
    best_robust = stats['std'].idxmin()
    
    print(f"📊 Highest Mean: {best_mean.upper()} ({stats.loc[best_mean, 'mean']:.4f})")
    print(f"🎯 Most Robust: {best_robust.upper()} (std = {stats.loc[best_robust, 'std']:.4f})")
    
    if best_mean == best_robust:
        print(f"\n✅ **{best_mean.upper()} wins on both metrics**")

In [ ]:
if not component_runs.empty:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Box plot
    variants = ['patchtst', 'gnn', 'ensemble']
    data = [component_runs[component_runs['ablation'] == v]['best_metric'].values 
            for v in variants if v in component_runs['ablation'].values]
    labels = [v.upper() for v in variants if v in component_runs['ablation'].values]
    colors = ['#3498db', '#e74c3c', '#2ecc71']
    
    bp = axes[0].boxplot(data, labels=labels, patch_artist=True, widths=0.6)
    for patch, color in zip(bp['boxes'], colors[:len(data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    axes[0].set_ylabel('Test Rank IC', fontsize=12, fontweight='bold')
    axes[0].set_title('Distribution Across 8 Held-Out Regimes', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Mean ± std
    stats = component_runs.groupby('ablation')['best_metric'].agg(['mean', 'std']).loc[
        [v for v in variants if v in component_runs['ablation'].values]
    ]
    
    x = np.arange(len(stats))
    bars = axes[1].bar(x, stats['mean'], yerr=stats['std'], 
                       color=colors[:len(stats)], alpha=0.7, capsize=8, 
                       edgecolor='black', linewidth=1.5)
    
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([a.upper() for a in stats.index])
    axes[1].set_ylabel('Mean Rank IC', fontsize=12, fontweight='bold')
    axes[1].set_title('Mean ± Std Dev', fontsize=13, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='y')
    
    for bar, mean, std in zip(bars, stats['mean'], stats['std']):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.01,
                    f'{mean:.4f}\n±{std:.4f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()

### Regime-Specific Performance Heatmap

In [ ]:
if not component_runs.empty and 'holdout_regime' in component_runs.columns:
    pivot = component_runs.pivot_table(
        index='ablation', columns='holdout_regime', values='best_metric', aggfunc='mean'
    )
    
    plt.figure(figsize=(14, 5))
    sns.heatmap(pivot, annot=True, fmt='.4f', cmap='RdYlGn', center=0.4,
                cbar_kws={'label': 'Test Rank IC'}, linewidths=1, linecolor='black')
    plt.title('Regime-Specific Generalization', fontsize=14, fontweight='bold', pad=15)
    plt.xlabel('Held-Out Regime', fontsize=12, fontweight='bold')
    plt.ylabel('Architecture', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

---

## Learnable Edge Weights Analysis

In [ ]:
ensemble_runs = all_runs[all_runs['ablation'] == 'ensemble']
fixed = ensemble_runs[ensemble_runs['learnable_edges'] == False]
learnable = ensemble_runs[ensemble_runs['learnable_edges'] == True]

if fixed.empty or learnable.empty:
    print("⚠️  Need both fixed and learnable edge runs.")
else:
    print("=" * 70)
    print("LEARNABLE EDGE WEIGHTS - ROBUSTNESS ANALYSIS")
    print("=" * 70)
    
    print(f"\n  Fixed:     {fixed['best_metric'].mean():.4f} ± {fixed['best_metric'].std():.4f}")
    print(f"  Learnable: {learnable['best_metric'].mean():.4f} ± {learnable['best_metric'].std():.4f}")
    
    mean_delta = learnable['best_metric'].mean() - fixed['best_metric'].mean()
    var_reduction = (1 - learnable['best_metric'].std() / fixed['best_metric'].std()) * 100
    
    print(f"\n  Δ Performance: {mean_delta:+.4f} ({(mean_delta/fixed['best_metric'].mean())*100:+.2f}%)")
    print(f"  Variance Reduction: {var_reduction:+.1f}%")
    
    if mean_delta > 0 and var_reduction > 0:
        print("\n✅ Learnable edges WIN (better + more robust)")

In [ ]:
if not fixed.empty and not learnable.empty and len(fixed) == len(learnable) == 8:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Box plot
    data = [fixed['best_metric'].values, learnable['best_metric'].values]
    bp = axes[0].boxplot(data, labels=['Fixed', 'Learnable'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][1].set_facecolor('#e74c3c')
    for box in bp['boxes']: box.set_alpha(0.7)
    axes[0].set_ylabel('Test Rank IC', fontsize=12, fontweight='bold')
    axes[0].set_title('Distribution', fontsize=13, fontweight='bold')
    axes[0].grid(True, alpha=0.3)
    
    # Parity plot
    fixed_sorted = fixed.sort_values('holdout_regime')['best_metric'].values
    learnable_sorted = learnable.sort_values('holdout_regime')['best_metric'].values
    axes[1].scatter(fixed_sorted, learnable_sorted, s=150, alpha=0.7, c='#9b59b6', edgecolors='black', linewidth=2)
    lims = [min(fixed_sorted.min(), learnable_sorted.min()) - 0.02,
            max(fixed_sorted.max(), learnable_sorted.max()) + 0.02]
    axes[1].plot(lims, lims, 'k--', alpha=0.5, linewidth=2, label='Parity')
    axes[1].set_xlabel('Fixed', fontsize=12, fontweight='bold')
    axes[1].set_ylabel('Learnable', fontsize=12, fontweight='bold')
    axes[1].set_title('Regime-by-Regime', fontsize=13, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Difference histogram
    diff = learnable_sorted - fixed_sorted
    axes[2].hist(diff, bins=8, alpha=0.7, edgecolor='black', color='#95a5a6', linewidth=1.5)
    axes[2].axvline(0, color='red', linestyle='--', linewidth=2)
    axes[2].axvline(diff.mean(), color='blue', linestyle='-', linewidth=2.5, label=f'Mean: {diff.mean():+.4f}')
    axes[2].set_xlabel('Δ (Learnable - Fixed)', fontsize=12, fontweight='bold')
    axes[2].set_ylabel('Frequency', fontsize=12, fontweight='bold')
    axes[2].set_title('Differences', fontsize=13, fontweight='bold')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

---

## Final Summary

In [ ]:
print("=" * 80)
print("CROSS-REGIME ABLATION STUDY: FINAL SUMMARY")
print("=" * 80)
print(f"\nData: 2018-05-01 to 2026-02-28 (~8 years)")
print(f"Method: Leave-One-Regime-Out CV (8 folds)")
print(f"Total Runs: {len(all_runs)}")

component_runs = all_runs[all_runs['ablation'].isin(['patchtst', 'gnn', 'ensemble'])]
if not component_runs.empty:
    print("\n" + "=" * 80)
    print("COMPONENT ABLATION")
    print("=" * 80)
    comp_stats = component_runs.groupby('ablation')['best_metric'].agg(['mean', 'std']).sort_values('mean', ascending=False)
    for rank, (variant, row) in enumerate(comp_stats.iterrows(), 1):
        print(f"  {rank}. {variant.upper():10} {row['mean']:.4f} ± {row['std']:.4f}")
    print(f"\n✅ Best: {comp_stats.index[0].upper()}")

if not fixed.empty and not learnable.empty:
    print("\n" + "=" * 80)
    print("LEARNABLE EDGE WEIGHTS")
    print("=" * 80)
    print(f"  Fixed:     {fixed['best_metric'].mean():.4f} ± {fixed['best_metric'].std():.4f}")
    print(f"  Learnable: {learnable['best_metric'].mean():.4f} ± {learnable['best_metric'].std():.4f}")
    mean_delta = learnable['best_metric'].mean() - fixed['best_metric'].mean()
    var_reduction = (1 - learnable['best_metric'].std() / fixed['best_metric'].std()) * 100
    if mean_delta > 0 and var_reduction > 0:
        print("\n✅ Recommendation: Use learnable edges")
    else:
        print("\n⚠️  Recommendation: Use fixed edges")

print("\n" + "=" * 80)
print("KEY TAKEAWAYS")
print("=" * 80)
print("""
1. Generalization > In-Sample Performance
2. Robustness (low variance) = reliable in production
3. 8-fold CV provides statistical confidence
4. Use best variant for deployment (retrain on full 2018-2026)
""")
print("=" * 80)